In [ ]:
import pandas as pd 
import numpy as np 

from ifc.config import load_data_config, load_split_config, resolve_project_root
from ifc.split import apply_forecasting_holdout_split, build_forecasting_test_frame

In [2]:

data_cfg = load_data_config() 
split_cfg = load_split_config()  
train_path, test_path = data_cfg.resolve_paths()    
print(data_cfg,"\n")

print(f'training path:{train_path}, exists? {train_path.exists()}')
print(f'tets path:{test_path}, exists? {test_path.exists()}')

id_col, time_col,drop_cols, target = data_cfg.id_col, data_cfg.time_col,data_cfg.drop_cols, data_cfg.target_col

print(f"Company_id column: {id_col}\nFiscal_year column: {time_col}\nTarget column: {target}")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)


DataConfig(train_path=WindowsPath('data/processed/train_data.csv'), test_path=WindowsPath('data/processed/test_features.csv'), id_col='company_id', time_col='fiscal_year', target_col='revenue_change', drop_cols=['bankruptcy_next_year', 'financial_health_class']) 

training path:C:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\data\processed\train_data.csv, exists? True
tets path:C:\Users\morio\OneDrive\Desktop\Projects\italian-financial-challenge\data\processed\test_features.csv, exists? True
Company_id column: company_id
Fiscal_year column: fiscal_year
Target column: revenue_change
train_df: (11828, 30)
test_df : (5811, 27)


In [22]:
import numpy as np
import pandas as pd

id_col = "company_id"
time_col = "fiscal_year"
target_t = "production_value"   # y(t)

# df_raw_train = il tuo train "raw" (non forecasting frame), anni 2018-2021
df0 = train_df.sort_values([id_col, time_col]).copy()

# 1) seleziona SOLO colonne numeriche di feature (escludi id/time e il target)
num_cols = df0.select_dtypes(include="number").columns.tolist()
num_feat_cols = [c for c in num_cols if c not in [time_col, target_t]]  # id_col non è numerico

# 2) crea le feature laggate (t-1)
for c in num_feat_cols:
    df0[f"lag1_{c}"] = df0.groupby(id_col)[c].shift(1)

# 3) tieni solo righe dove esiste lag e target
lag_cols = [f"lag1_{c}" for c in num_feat_cols]
df_corr = df0.dropna(subset=lag_cols + [target_t], how="any").copy()

# 4) correlazione tra X(t-1) e y(t)
pearson = df_corr[lag_cols].corrwith(df_corr[target_t], method="pearson").sort_values(key=lambda s: s.abs(), ascending=False)
spearman = df_corr[lag_cols].corrwith(df_corr[target_t], method="spearman").sort_values(key=lambda s: s.abs(), ascending=False)

print("Top Pearson:")
print(pearson.head(20))

print("\nTop Spearman:")
print(spearman.head(20))


Top Pearson:
lag1_production_costs       0.143918
lag1_current_assets         0.131927
lag1_financial_expenses     0.123043
lag1_total_assets           0.117710
lag1_shareholders_equity    0.116452
lag1_total_debt             0.116125
lag1_short_term_debt        0.114373
lag1_long_term_debt         0.114194
lag1_operating_income       0.107889
lag1_net_profit_loss        0.100989
lag1_total_fixed_assets     0.092835
lag1_financial_income       0.071011
lag1_current_ratio          0.017929
lag1_quick_ratio            0.017927
lag1_profit_margin         -0.014659
lag1_roe                    0.009419
lag1_roi                    0.009251
lag1_revenue_change         0.007951
lag1_years_in_business      0.005077
lag1_leverage              -0.002028
dtype: float64

Top Spearman:
lag1_production_costs       0.306876
lag1_current_assets         0.259095
lag1_operating_income       0.249141
lag1_total_assets           0.244916
lag1_total_debt             0.242521
lag1_short_term_debt        0.24

In [3]:
train_df = train_df.drop(drop_cols,axis=1)

def ateco_to_string(df, col="ateco_sector"):
    df = df.copy()
    df[col] = df[col].astype("Int64").astype("object")
    return df

train_df = ateco_to_string(train_df)

In [4]:
df_train, df_val = apply_forecasting_holdout_split(train_df, split_cfg, drop_cols=data_cfg.drop_cols)
df_test_forecast = build_forecasting_test_frame(train_df, test_df, split_cfg, drop_cols=data_cfg.drop_cols)

print("df_train_forecast:", df_train.shape, "years:", sorted(df_train[time_col].unique()))
print("df_val_forecast  :", df_val.shape, "years:", sorted(df_val[time_col].unique()))
print("df_test_forecast :", df_test_forecast.shape, "years:", sorted(df_test_forecast[time_col].unique()))

df_train_forecast: (5897, 28) years: [np.int64(2019), np.int64(2020)]
df_val_forecast  : (2932, 28) years: [np.int64(2021)]
df_test_forecast : (5811, 27) years: [np.int64(2022), np.int64(2023)]


In [21]:
num = df_train.select_dtypes(include='number')

corr = num.corr(method='spearman')[target].sort_values(ascending=False)

print(corr)

revenue_change              1.000000
prev_debt_to_assets         0.003992
prev_leverage               0.003986
prev_years_in_business     -0.001333
fiscal_year                -0.002332
prev_profit_margin         -0.009736
prev_current_ratio         -0.012512
prev_quick_ratio           -0.012513
prev_roe                   -0.067498
prev_roi                   -0.070502
prev_net_profit_loss       -0.440390
prev_financial_income      -0.468168
prev_operating_income      -0.491601
prev_total_fixed_assets    -0.520032
prev_shareholders_equity   -0.531806
prev_financial_expenses    -0.540655
prev_long_term_debt        -0.544527
prev_short_term_debt       -0.545829
prev_total_debt            -0.550138
prev_total_assets          -0.552095
prev_current_assets        -0.554395
prev_production_costs      -0.560242
prev_production_value      -0.561449
Name: revenue_change, dtype: float64


In [5]:
X_train = df_train.drop([id_col,time_col,target],axis=1).copy()
y_train = df_train[target].astype(float)

X_test = df_val.drop([id_col,time_col,target],axis=1).copy()
y_test = df_val[target].astype(float)



In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [7]:


from xgboost import XGBRegressor  # pip install xgboost

# ----------------------------
# 1) Signed-log target transform
# ----------------------------
def slog(y):
    y = np.asarray(y, dtype=float)
    return np.sign(y) * np.log1p(np.abs(y))

def inv_slog(z):
    z = np.asarray(z, dtype=float)
    return np.sign(z) * (np.expm1(np.abs(z)))



# ----------------------------
# 3) Preprocess: numeric + categorical
# ----------------------------
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # min_frequency rende l'one-hot "più light" (meno colonne rare)
    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=True)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

# ----------------------------
# 4) Light XGBoost model
# ----------------------------
xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=600,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

model = TransformedTargetRegressor(
    regressor=Pipeline(steps=[
        ("prep", preprocess),
        ("xgb", xgb),
    ]),
    func=slog,
    inverse_func=inv_slog,
)

# ----------------------------
# 5) Fit + eval
# ----------------------------
model.fit(X_train, y_train)
pred_val = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred_val))
mae = mean_absolute_error(y_test, pred_val)

# MAPE "safe" (attenzione: vicino a 0 esplode)
mape_safe = np.mean(np.abs((y_test - pred_val) / np.clip(np.abs(y_test), 1e-6, None)))

dir_acc = np.mean(np.sign(y_test.values) == np.sign(pred_val))

print(f"RMSE: {rmse:,.3f}")
print(f"MAE:  {mae:,.3f}")
print(f"MAPE_safe: {mape_safe:,.3f}")
print(f"Directional Acc: {dir_acc:,.3f}")


RMSE: 6,165.273
MAE:  517.056
MAPE_safe: 1.651
Directional Acc: 0.742


In [8]:
R2 = r2_score(y_test,pred_val)
print(R2)

0.037389890969470874


In [9]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

def eval_metrics(y, yhat):
    rmse = np.sqrt(mean_squared_error(y, yhat))
    mae = mean_absolute_error(y, yhat)
    r2 = r2_score(y, yhat)
    dir_acc = np.mean(np.sign(y) == np.sign(yhat))
    return rmse, mae, r2, dir_acc

# baseline: predici 0
yhat0 = np.zeros_like(y_test, dtype=float)
print("Zero baseline:", eval_metrics(y_test.values, yhat0))

# baseline: predici mediana train
yhat_med = np.full_like(y_test.values, np.median(y_train.values), dtype=float)
print("Median baseline:", eval_metrics(y_test.values, yhat_med))


Zero baseline: (np.float64(6302.851267132681), 549.6354160982265, -0.006050783113765723, np.float64(0.0))
Median baseline: (np.float64(6302.766054787306), 549.6068553888131, -0.006023580387997107, np.float64(0.5133015006821282))


In [10]:
y = y_test.values

dir_pos = np.mean(np.sign(y) == 1)   # baseline che predice sempre + (crescita)
dir_neg = np.mean(np.sign(y) == -1)  # baseline che predice sempre - (calo)

print("Dir baseline always +:", dir_pos)
print("Dir baseline always -:", dir_neg)
print("Dir baseline best:", max(dir_pos, dir_neg))


Dir baseline always +: 0.5133015006821282
Dir baseline always -: 0.48669849931787174
Dir baseline best: 0.5133015006821282


In [11]:
lo, hi = np.quantile(y_test.values, [0.01, 0.99])
rmse_clip = np.sqrt(mean_squared_error(np.clip(y_test.values, lo, hi),
                                       np.clip(pred_val, lo, hi)))
mae_clip  = mean_absolute_error(np.clip(y_test.values, lo, hi),
                                np.clip(pred_val, lo, hi))
print(rmse_clip, mae_clip)


782.8043522572257 320.2845140569267


light xgb with pseudo-huber ad flag `production_value` < p10

apply windorization 

In [20]:
import numpy as np

lo_q, hi_q = 0.01, 0.99

lo = np.quantile(y_train.values, lo_q)
hi = np.quantile(y_train.values, hi_q)

y_train_w = np.clip(y_train.values, lo, hi)

# fit sul target winsorizzato
model = Pipeline(steps=[
    ("feat", feat),         # il tuo SmallP10Flagger su X
    ("prep", preprocess),
    ("xgb", xgb),
])

model.fit(X_train, y_train_w)

pred = model.predict(X_test)

# VALUTAZIONE: sempre su y_test ORIGINALE (no leakage)
rmse = np.sqrt(mean_squared_error(y_test.values, pred))
mae  = mean_absolute_error(y_test.values, pred)
r2   = r2_score(y_test.values, pred)

print(lo, hi)
print(rmse, mae, r2)


-98.44 6972.600399999999
6270.844983523088 501.16630950688057 0.004140855124192266


model performance drop signed-log is the best choice

In [28]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class SmallP10Flagger(BaseEstimator, TransformerMixin):
    def __init__(self, col="prev_production_value", q=0.10, add_log=True, suffix="pv"):
        self.col = col
        self.q = q
        self.add_log = add_log
        self.suffix = suffix

    def fit(self, X, y=None):
        Xdf = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.p10_ = Xdf[self.col].quantile(self.q)
        return self

    def transform(self, X):
        Xdf = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X).copy()
        x = Xdf[self.col]

        Xdf[f"is_small_{self.suffix}_p10"] = ((x <= self.p10_) & x.notna()).astype("int8")
        Xdf[f"{self.suffix}_missing"] = x.isna().astype("int8")

        if self.add_log:
            Xdf[f"log_{self.suffix}"] = np.log1p(x.fillna(0))

        return Xdf


In [27]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5897 entries, 1 to 11826
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   prev_province             5439 non-null   object 
 1   prev_region               5897 non-null   object 
 2   prev_ateco_sector         5897 non-null   object 
 3   prev_legal_form           5897 non-null   object 
 4   prev_years_in_business    5897 non-null   float64
 5   prev_total_fixed_assets   5897 non-null   float64
 6   prev_current_assets       5897 non-null   float64
 7   prev_total_assets         5897 non-null   float64
 8   prev_shareholders_equity  5897 non-null   float64
 9   prev_total_debt           5897 non-null   float64
 10  prev_short_term_debt      5897 non-null   float64
 11  prev_long_term_debt       5897 non-null   float64
 12  prev_production_value     5897 non-null   float64
 13  prev_production_costs     5897 non-null   float64
 14  prev_operati

In [32]:
def slog(y):
    y = np.asarray(y, dtype=float)
    return np.sign(y) * np.log1p(np.abs(y))

def inv_slog(z):
    z = np.asarray(z, dtype=float)
    return np.sign(z) * (np.expm1(np.abs(z)))

feat = SmallP10Flagger(col="prev_production_value", q=0.10, add_log=True, suffix="prev_pv")

# colonne base da X_train
extra_cols = ["log_prev_pv", "is_small_prev_pv_p10", "prev_pv_missing"]

num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(include="object").columns.tolist()

# includi le nuove feature (create dal transformer feat)
num_cols = num_cols + extra_cols

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=True)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

xgb = XGBRegressor(
    objective="reg:pseudohubererror",
    n_estimators=600,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
)

slog_model = TransformedTargetRegressor(
    regressor=Pipeline(steps=[
        ("feat", feat),
        ("prep", preprocess),
        ("xgb", xgb),
    ]),
    func=slog,
    inverse_func=inv_slog,
)

In [34]:
slog_model.fit(X_train, y_train)
pred_val = slog_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred_val))
mae = mean_absolute_error(y_test, pred_val)
R2 = r2_score(y_test,pred_val)

mape = mean_absolute_percentage_error(y_test,pred_val)
dir_acc = np.mean(np.sign(y_test.values) == np.sign(pred_val))

print(f"RMSE: {rmse:,.3f}")
print(f"MAE:  {mae:,.3f}")
print(f"MAPE_safe: {mape_safe:,.3f}")
print(f"Directional Acc: {dir_acc:,.3f}")
print(f"R2 score:{R2}")
print(f"mape:{mape}")

RMSE: 5,591.289
MAE:  502.557
MAPE_safe: 2.743
Directional Acc: 0.744
R2 score:0.20828367293467753
mape:2.743347733840112


build `production_value_ratio` = `pv_t-1` / `pv_t-2`